In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt

from functools import partial

import sys
sys.path.append('../src')

from load_clean_data import load_clean_data
from backtest import get_cutoffs, run_backtest
from features import build_features, FEATURE_COLUMNS
from evaluation import score_forecasts, merge_backtests
from models import add_lgbm_forecast, LGBM_PARAMS, add_xgb_forecast, XGB_PARAMS
from tuning import random_search

In [2]:
df = load_clean_data()
df_with_features = build_features(df)

In [3]:
cutoffs = get_cutoffs(df['Date'].max(), 7)
first_10 = cutoffs[:10]

In [4]:
XGB_SEARCH_SPACE = {'max_depth': ('int', 2, 10),
                    'learning_rate': ('log', 0.01, 0.3),
                    'n_estimators': ('int', 100, 600),
                    'min_child_weight': ('int', 1, 50),
                    'subsample': ('float', 0.5, 1.0),
                    'colsample_bytree': ('float', 0.5, 1.0),
                    'reg_lambda': ('log', 0.1, 10)
                    }

LGBM_SEARCH_SPACE = {'num_leaves': ('int', 4, 127),
                     'learning_rate': ('log', 0.01, 0.3),
                     'n_estimators': ('int', 100, 600),
                     'min_child_samples': ('int', 10, 200),
                     'subsample': ('float', 0.5, 1.0),
                     'colsample_bytree': ('float', 0.5, 1.0),
                     'reg_lambda': ('log', 0.1, 10)
                    }

In [5]:
RUN_XGB_SEARCH = False

In [6]:
if RUN_XGB_SEARCH:
    add_xgb_with_storeid = partial(add_xgb_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])
    xgb_tuning = random_search(df_with_features, first_10, add_xgb_with_storeid, 'xgb_forecast', XGB_PARAMS, XGB_SEARCH_SPACE, n_trials=25, path='../results/tuning/xgb_storeid_tuning.csv')

    print(xgb_tuning.head(5))

else:
    xgb_tuning = pd.read_csv('../results/tuning/xgb_storeid_tuning.csv')

    print(xgb_tuning.head(5))

   trial  max_depth  learning_rate  n_estimators  min_child_weight  subsample  \
0     25         10       0.032536           391              31.0   0.511402   
1      6          9       0.016901           472              35.0   0.872381   
2      4          9       0.012424           530              42.0   0.815832   
3     17          8       0.054967           180              47.0   0.848160   
4     15          5       0.066497           431              26.0   0.832157   

   colsample_bytree  reg_lambda        RMSE         MAE      WAPE     seconds  
0          0.979280    0.921737  973.229736  545.247192  9.438339  380.053567  
1          0.983755    0.448385  984.236206  555.037720  9.607815  347.696682  
2          0.879044    0.511743  986.777161  557.520142  9.650786  365.448849  
3          0.723078    0.578153  987.265137  559.125671  9.678578  105.586660  
4          0.703193    4.246594  989.486694  560.889465  9.709110  139.961296  


In [7]:
RUN_LGBM_SEARCH = False

In [8]:
if RUN_LGBM_SEARCH:
    add_lgbm_with_storeid = partial(add_lgbm_forecast, feature_cols = FEATURE_COLUMNS + ['StoreID'])
    lgbm_tuning = random_search(df_with_features, first_10, add_lgbm_with_storeid, 'lgbm_forecast', LGBM_PARAMS, LGBM_SEARCH_SPACE, n_trials=25, path='../results/tuning/lgbm_storeid_tuning.csv')

    print(lgbm_tuning.head(5))

else:
    lgbm_tuning = pd.read_csv('../results/tuning/lgbm_storeid_tuning.csv')

    print(lgbm_tuning.head(5))

   trial  num_leaves  learning_rate  n_estimators  min_child_samples  \
0     11          38       0.047629           491              106.0   
1     17          92       0.054967           180              189.0   
2     12          74       0.049644           528              117.0   
3     25         126       0.032536           391              126.0   
4      7         116       0.049385           285              161.0   

   subsample  colsample_bytree  reg_lambda         RMSE         MAE      WAPE  \
0   0.569898          0.557265    2.171730   991.206942  561.487773  9.719466   
1   0.848160          0.723078    0.578153   994.485315  566.938010  9.813811   
2   0.882499          0.817359    1.279849   992.052071  567.497825  9.823501   
3   0.511402          0.979280    0.921737  1002.771018  569.931384  9.865627   
4   0.564961          0.737852    0.284327   999.111324  570.495134  9.875385   

     seconds  
0  84.865292  
1  42.138759  
2  93.057771  
3  94.964417  
4  67

In [9]:
XGB_TUNED_PARAMS = {
    'tree_method': 'hist',
    'enable_categorical': True,
    'n_estimators': 391,
    'learning_rate': 0.032536344732155796,
    'max_depth': 10,
    'min_child_weight': 31.0,
    'subsample': 0.5114019355148487,
    'colsample_bytree': 0.9792796066207227,
    'reg_lambda': 0.9217366875216192,
    'random_state': 42
}

LGBM_TUNED_PARAMS = {
    'n_estimators': 491,
    'learning_rate': 0.04762930519509523,
    'num_leaves': 38,
    'subsample_freq': 1,
    'min_child_samples': 106,
    'subsample': 0.5698984990638287,
    'colsample_bytree': 0.5572650367679868,
    'reg_lambda': 2.171730464570453,
    'random_state': 42,
    'verbose': -1
}

In [10]:
FORECAST = False

if FORECAST:
    add_tuned_xgb = partial(add_xgb_forecast, params=XGB_TUNED_PARAMS, feature_cols = FEATURE_COLUMNS + ['StoreID'])

    bt = run_backtest(df_with_features, cutoffs, 7, add_tuned_xgb)

    xgb_tuned_bt = bt.rename(columns={'xgb_forecast': 'xgb_storeid_tuned_forecast'})

    xgb_tuned_bt = xgb_tuned_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'xgb_storeid_tuned_forecast']]

    xgb_tuned_bt.to_csv('../results/backtests/xgb_storeid_tuned_backtest.csv', index=False)

else: 
    xgb_tuned_bt = pd.read_csv('../results/backtests/xgb_storeid_tuned_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [11]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(xgb_tuned_bt, ['xgb_storeid_tuned_forecast'])
    scores.to_csv('../results/xgb_storeid_tuned_scores.csv', index=False)

In [12]:
FORECAST = True

if FORECAST:
    add_tuned_lgbm = partial(add_lgbm_forecast, params=LGBM_TUNED_PARAMS, feature_cols = FEATURE_COLUMNS + ['StoreID'])

    bt = run_backtest(df_with_features, cutoffs, 7, add_tuned_lgbm)

    lgbm_tuned_bt = bt.rename(columns={'lgbm_forecast': 'lgbm_storeid_tuned_forecast'})

    lgbm_tuned_bt = lgbm_tuned_bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'lgbm_storeid_tuned_forecast']]

    lgbm_tuned_bt.to_csv('../results/backtests/lgbm_storeid_tuned_backtest.csv', index=False)

else: 
    lgbm_tuned_bt = pd.read_csv('../results/backtests/lgbm_storeid_tuned_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [13]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(lgbm_tuned_bt, ['lgbm_storeid_tuned_forecast'])
    scores.to_csv('../results/lgbm_storeid_tuned_scores.csv', index=False)

In [16]:
combined_bt = merge_backtests(['baselines', 'xgb', 'lgbm', 'xgb_tuned', 'lgbm_tuned', 'xgb_storeid', 'lgbm_storeid', 'xgb_storeid_tuned', 'lgbm_storeid_tuned'])

forecasts = ['naive_baseline', 'avg_28d_baseline', 'avg_all_baseline', 
             'xgb_forecast', 'xgb_tuned_forecast', 'xgb_storeid_forecast', 'xgb_storeid_tuned_forecast', 
             'lgbm_forecast', 'lgbm_tuned_forecast', 'lgbm_storeid_forecast', 'lgbm_storeid_tuned_forecast',
             ]

combined_scores = score_forecasts(combined_bt, forecasts)

model_names = {
    'naive_baseline': 'Naive Baseline',
    'avg_28d_baseline': '28-Day Average Baseline',
    'avg_all_baseline': 'Full History Average Baseline',
    'xgb_forecast': 'XGBoost Forecast',
    'xgb_tuned_forecast': 'XGBoost Tuned',
    'xgb_storeid_forecast': 'XGBoost with StoreID',
    'xgb_storeid_tuned_forecast': 'XGBoost with StoreID Tuned',
    'lgbm_forecast': 'LightGBM Forecast',
    'lgbm_tuned_forecast': 'LightGBM Tuned',
    'lgbm_storeid_forecast': 'LightGBM with StoreID',
    'lgbm_storeid_tuned_forecast': 'LightGBM with StoreID Tuned'
}

relabelled_scores = combined_scores.copy()

relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
XGBoost with StoreID Tuned,907.47,541.47,9.24%
XGBoost with StoreID,922.86,555.70,9.48%
LightGBM with StoreID Tuned,940.50,564.84,9.64%
LightGBM with StoreID,961.10,580.75,9.91%
LightGBM Tuned,994.37,597.28,10.19%
XGBoost Forecast,1001.30,604.95,10.33%
XGBoost Tuned,992.88,605.63,10.34%
LightGBM Forecast,1035.76,619.47,10.57%
Full History Average Baseline,2029.48,1261.49,21.53%
28-Day Average Baseline,2127.55,1326.10,22.63%


In [17]:
first_10 = cutoffs[:10]
last_15_bt = combined_bt[~combined_bt['cutoff'].isin(first_10)]

assert last_15_bt['cutoff'].nunique() == 15, 'not 15 cutoffs'
assert len(last_15_bt) == 117075, 'wrong length'

last_15_scores = score_forecasts(last_15_bt, forecasts)

relabelled_scores = last_15_scores.copy()

relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
XGBoost with StoreID Tuned,860.84,538.95,9.11%
XGBoost with StoreID,870.71,551.70,9.33%
LightGBM with StoreID Tuned,905.12,567.07,9.59%
LightGBM with StoreID,915.32,575.86,9.74%
LightGBM Tuned,949.23,593.35,10.03%
XGBoost Forecast,956.45,599.60,10.14%
XGBoost Tuned,961.86,608.98,10.30%
LightGBM Forecast,1011.97,620.17,10.49%
Full History Average Baseline,2019.27,1278.61,21.62%
28-Day Average Baseline,2151.14,1368.45,23.14%
